In [ ]:
from dask.distributed import Client, LocalCluster

client = Client(n_workers=3, threads_per_worker=1)
client

In [ ]:
import copy
import sys
import xarray as xr
import numpy as np
import dask.array as da

import matplotlib.pyplot as plt
import hvplot.xarray
import scipy.constants
import datetime

sys.path.append("../..")
import processing_dask as pr
import plot_dask
import processing as old_processing
import px4_log_loader

sys.path.append("../../../preprocessing/")
from generate_chirp import generate_chirp

In [ ]:
base_path = "/home/thomas/Documents/StanfordGrad/RadioGlaciology/"

# Slakbreen 3

# prefix = base_path + "drone/radar_data/20230315-slakbreen-day3/20230315_064228" # Flight 1
# px4_logs = base_path + "px4_logs/2023-03-15/13_35_40.ulg"

# prefix = base_path + "drone/radar_data/20230315-slakbreen-day3/20230315_070941" # Flight 2
# px4_logs = base_path + "px4_logs/2023-03-15/14_03_44.ulg" # Clipping a little, but doesn't seem too bad

prefix = base_path + "drone/radar_data/20230315-slakbreen-day3/20230315_074655" # Flight 3
px4_logs = base_path + "px4_logs/2023-03-15/14_38_58.ulg"

In [ ]:
# resave data as zarr for dask processing
zarr_base_location=base_path + "test_tmp_zarr_cache/"
zarr_path = pr.save_radar_data_to_zarr(prefix, zarr_base_location=zarr_base_location, skip_if_cached=True)

# open zarr file, adjust chunk size to be 10 MB - 1 GB based on sample rate/bit depth
raw = xr.open_zarr(zarr_path, chunks={"pulse_idx": 1000})
raw = pr.construct_slow_time_vectors(raw, allow_reading_data=True)

In [ ]:
# Open the PX4 log file
px4_data = px4_log_loader.ulog_to_xarray(px4_logs,
                                         message_types_whitelist=[
                                             ("vehicle_gps_position", 0),
                                             ("distance_sensor", 0),
                                             ('vehicle_global_position', 0),
                                             ('vehicle_local_position', 0),
                                             ('vehicle_attitude', 0),
                                             ('wind', 0)
                                             ])

for key in px4_data.keys():
    px4_data[key] = px4_data[key].interp(timestamp_utc = raw.timestamp, method='nearest')

In [ ]:
distance_agl_filt = px4_data[('distance_sensor', 0)]['current_distance'].copy()

# Remove anything with reported zero signal quality
distance_agl_filt[px4_data[('distance_sensor', 0)]['signal_quality'] == 0] = np.nan
# Remove anything where the reported distance is implausibly low
distance_agl_filt[distance_agl_filt < 5] = np.nan

for j in range(1000):
    distance_agl_filt = distance_agl_filt.dropna(dim='pulse_idx')
    noisy_diff_mask = distance_agl_filt.diff(dim='pulse_idx') < -10
    distance_agl_filt[1:][noisy_diff_mask] = np.nan

# Rolling average
distance_agl_filt = distance_agl_filt.rolling(pulse_idx=400, center=True).median(skipna=True)
distance_agl_filt = distance_agl_filt.dropna(dim='pulse_idx')

fig, ax = plt.subplots(figsize=(10, 5))
distance_agl_filt.plot(x='slow_time', marker=".", linewidth=1, ax=ax)
ax.set_ylabel("Filtered distance AGL [m]")
plt.show()

In [ ]:
zero_sample_idx = 159 # B205mini, fs = 56 MHz

nstack = 20 # number of pulses to stack

modify_rx_window = True # set to true if you want to window the reference chirp only on receive, false uses ref chirp as transmitted in config file
rx_window = "kaiser14" # what you want to change the rx window to if modify_rx_window is true

#dielectric_constant = 1 # air
dielectric_constant = 3.17 # ice (air = 1, 66% velocity coax = 2.2957)
#dielectric_constant = 2.2957 # COAX (air = 1, 66% velocity coax = 2.2957)
sig_speed = scipy.constants.c / np.sqrt(dielectric_constant)

# Generate reference chirp

if modify_rx_window:
    config = copy.deepcopy(raw.config)
    config['GENERATE']['window'] = rx_window
else:
    config = raw.config

chirp_ts, ref_chirp = generate_chirp(config)

In [ ]:
def label_axis(ax, text_y=1.05):
    # Label plot with data source and processing
    ax.text(0, text_y, prefix.split("/")[-1] + "\n" + f"n_stack * num_presums = {nstack * raw.config['CHIRP']['num_presums']}", horizontalalignment='left', verticalalignment='center', transform=ax.transAxes, fontdict={'size': 8})

prefix_tag = prefix.split("/")[-1]

In [ ]:
stacked = pr.remove_errors(raw) # Remove errors
stacked = pr.stack(stacked, nstack) # stack

compressed = pr.pulse_compress(stacked, ref_chirp,
                               fs=stacked.config['GENERATE']['sample_rate'],
                               zero_sample_idx=zero_sample_idx,
                               signal_speed=sig_speed)

compressed_power = xr.apply_ufunc(
    lambda x: 20*np.log10(np.abs(x)),
    compressed,
    dask="parallelized"
)

In [ ]:
compressed_power.timestamp

In [ ]:
px4_data[('vehicle_local_position', 0)]

In [ ]:
subsample = 10
vehicle_pos_ds = px4_data[('vehicle_local_position', 0)].isel(pulse_idx=slice(None, None, subsample))

nanmask = np.isnan(vehicle_pos_ds.x) | np.isnan(vehicle_pos_ds.y)
vehicle_pos_ds = vehicle_pos_ds.where(~nanmask, drop=True)

dx = np.diff(vehicle_pos_ds.x)
dy = np.diff(vehicle_pos_ds.y)
distances = np.sqrt(dx**2 + dy**2)
distance_travelled = np.cumsum(da.concatenate([[0], distances]))
vehicle_pos_ds['distance_travelled'] = (('pulse_idx'), distance_travelled)
distance_travelled = vehicle_pos_ds.distance_travelled.compute()
distance_travelled.plot()

In [ ]:
compressed_power['distance_travelled'] = distance_travelled.interp(pulse_idx=compressed_power.pulse_idx, method='linear')
compressed_power

In [ ]:
# Plot radargram
fig, ax = plt.subplots(1,1, figsize=(12,4), facecolor='white')

compressed_power_masked = compressed_power.dropna(dim='pulse_idx')

p = ax.pcolormesh(compressed_power_masked.distance_travelled/1e3, compressed_power_masked.reflection_distance, compressed_power_masked.radar_data.transpose(),
                  shading='auto', cmap='inferno', vmin=-75, vmax=-30)
ax.invert_yaxis()
clb = fig.colorbar(p, ax=ax)
clb.set_label('Return Power [dB]')
ax.set_xlabel('Distance [km]')
ax.set_ylabel('Distance to Reflector [m]')
# relevant options: ax.set_ylim=(100,-50), ax.set_xlim=(0, 1), vmin=-90, vmax=40
ax.set_ylim(200, 0)

label_axis(ax)
fig.tight_layout()
fig.savefig("example_radargram.png", dpi=300, bbox_inches='tight')

plt.show()